## End-to-end demo of CREST using MNIST data 

The entire notebook should run in < 60 seconds, and use ~600MB of memory.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ../..

#### Load MNIST

In [3]:
import tensorflow as tf
import numpy as np

# Load data and define constants
xy,xy_test = tf.keras.datasets.mnist.load_data() 
xy_train   = [i[:50000]for i in xy]
xy_valid   = [i[50000:]for i in xy]
n_classes  = 10
patch_size = 28

2023-07-10 21:15:29.356897: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


#### Create the datasets

In [4]:
from crest.src.data.loading import StructuredDataset

# Set input/output key labels
INP = 'image'
OUT = 'class'

# Define the train/test datasets, as well as the array labels
ds_train = StructuredDataset(*xy_train, labels=[INP, OUT])
ds_valid = StructuredDataset(*xy_valid, labels=[INP, OUT])
ds_test  = StructuredDataset(*xy_test,  labels=[INP, OUT])

/usr/local/anaconda3/envs/crest_cpu/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Create the batchers

In [5]:
from crest.src.data import Batcher

# Note: workers=0 is faster for this, but may cause issues in a notebook
# since the background threads will run in the main process 
batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}
batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)
batch_test  = Batcher(ds_test ,  **batch_kwargs)

#### Create the CREST graph

In [6]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.src.model import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='MNIST',
    inputs={INP : tf.TensorSpec(shape=[None,patch_size, patch_size])},
    outputs={OUT : tf.TensorSpec(shape=[None,10])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

[None, None, None, None, None, None, None]


#### Build, train, and test using Model

In [7]:
from crest import Model

build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'sparse_categorical_crossentropy', 
    'metrics'   : 'sparse_categorical_accuracy'
}

model = Model(graph)
model.build(**build_kwargs)
model.fit(batch_train, **{
    'validation_data'  : batch_valid,
    'steps_per_epoch'  : 250, # 50k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1,
})


2023-07-10 21:15:38.065847: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/5
250/250 [==============================] - 6s 22ms/step - loss: 4.0073 - sparse_categorical_accuracy: 0.8662 - val_loss: 0.5358 - val_sparse_categorical_accuracy: 0.9551
Epoch 2/5
250/250 [==============================] - 5s 21ms/step - loss: 0.5367 - sparse_categorical_accuracy: 0.9409 - val_loss: 0.2838 - val_sparse_categorical_accuracy: 0.9646
Epoch 3/5
250/250 [==============================] - 5s 19ms/step - loss: 0.2711 - sparse_categorical_accuracy: 0.9563 - val_loss: 0.1978 - val_sparse_categorical_accuracy: 0.9707
Epoch 4/5
250/250 [==============================] - 5s 19ms/step - loss: 0.1788 - sparse_categorical_accuracy: 0.9643 - val_loss: 0.1721 - val_sparse_categorical_accuracy: 0.9706
Epoch 5/5
250/250 [==============================] - 5s 19ms/step - loss: 0.1285 - sparse_categorical_accuracy: 0.9698 - val_loss: 0.1535 - val_sparse_categorical_accuracy: 0.9728


In [8]:
# Evaluate the model on the test set
accuracy = model.evaluate(batch_test,verbose=False,return_dict=True)
print(f'Test performance metrics: {accuracy}')

Test performance metrics: {'loss': 0.05012156441807747, 'sparse_categorical_accuracy': 0.9850000143051147}


Accuracy on the full test set should be > ~97%.

In [9]:
# Close the batchers
batch_train.close()
batch_valid.close()
batch_test.close()

### Make predictions

In [10]:
x_pred,_ = xy_test  # Use images from the testing set
ds_pred = StructuredDataset(*[x_pred], labels=[INP])
batch_pred = Batcher(ds_pred, **{
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
})

In [11]:
# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size' : 200,
    'verbose'    : False
}

In [12]:
# Generate predictions on the whole test set
pred_kwargs['steps'] = 10000//pred_kwargs['batch_size']
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
    'shuffle'    : False
}) 
model_pred_batch = model.predict(batch_pred, **pred_kwargs) # Using a batcher
model_pred_dataset = model.predict(ds_pred,**pred_kwargs)   # Using a dataset
print('Difference:', (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


Adding auxillary output like lat and lon when making a prediction

In [13]:
# Combine some fictous auxillary data with testing input for prediction
x,_ = xy_test
lat = np.arange(len(x),dtype=float)
lon = np.arange(len(x),dtype=float)
x_pred = (lat,lon,x)

In [14]:
# Create the prediction data and make predictions
ds_pred = StructuredDataset(*x_pred, labels=['lat','lon',INP])
aux_pred = model.predict(ds_pred,labels=['lat','lon'],**pred_kwargs)

In [15]:
# Check you have lat,lon now in your output
aux_pred 

{'class': array([[1.3477293e-20, 7.4430634e-22, 1.9162078e-16, ..., 1.0000000e+00,
         4.7179309e-12, 6.8821025e-12],
        [7.2871675e-10, 1.1804441e-09, 9.9999785e-01, ..., 8.5570243e-31,
         1.3992560e-12, 3.3938433e-24],
        [5.8142396e-10, 9.9999249e-01, 2.1475105e-07, ..., 3.6194236e-10,
         7.3122042e-06, 7.1902720e-11],
        ...,
        [4.8530237e-19, 6.5845807e-14, 9.8667673e-22, ..., 1.1940269e-06,
         2.6890970e-07, 1.8734294e-05],
        [2.7168741e-21, 2.5964637e-24, 2.0923378e-21, ..., 1.5700245e-16,
         3.3584097e-08, 1.1727806e-16],
        [2.3534043e-20, 2.1520310e-20, 4.2259215e-17, ..., 1.5629224e-20,
         5.7808924e-16, 7.8797904e-21]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}